<a href="https://colab.research.google.com/github/Superpevel/liya_diploma/blob/main/notebooks/01_dataset_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
One-shot Colab bootstrap for liya_diplomCC.

Run this in the FIRST cell of any notebook on Google Colab when you want a
clean fresh setup (clones the repo into Drive, installs deps, mounts, sets paths).

After running once per Colab runtime, each notebook's existing Cell 0 will
work as-is (it expects DRIVE_ROOT to point at the project inside MyDrive).

Usage:
    !curl -sSL https://raw.githubusercontent.com/YOUR_USERNAME/liya_diplomCC/main/setup_colab.py | python -

Or, after you've cloned the repo into Drive once:
    !python /content/drive/MyDrive/liya_diplomCC/setup_colab.py
"""

import os
import subprocess
import sys

GITHUB_URL = "https://github.com/superpevel/liya_diploma.git"  # EDIT THIS
DRIVE_ROOT = "/content/drive/MyDrive/liya_diplomCC"
AI_TOOLKIT = "/content/ai-toolkit"


def sh(cmd: str) -> int:
    print(f"$ {cmd}")
    return subprocess.call(cmd, shell=True)


def main():
    # Mount Drive
    if not os.path.ismount("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")

    # Clone or update the project in Drive
    if not os.path.exists(DRIVE_ROOT):
        sh(f"git clone {GITHUB_URL} {DRIVE_ROOT}")
    else:
        print(f"Project already in Drive at {DRIVE_ROOT}; running git pull.")
        sh(f"cd {DRIVE_ROOT} && git pull --ff-only")

    # Install Python deps
    sh(f"pip install -q -r {DRIVE_ROOT}/requirements.txt")

    # Clone & install ai-toolkit (LoRA training engine)
    if not os.path.exists(AI_TOOLKIT):
        sh(f"git clone https://github.com/ostris/ai-toolkit {AI_TOOLKIT}")
    sh(f"pip install -q -r {AI_TOOLKIT}/requirements.txt")

    # Make modules importable
    for p in (DRIVE_ROOT, AI_TOOLKIT):
        if p not in sys.path:
            sys.path.insert(0, p)

    print("\n=== Setup complete ===")
    print(f"DRIVE_ROOT: {DRIVE_ROOT}")
    print(f"AI_TOOLKIT: {AI_TOOLKIT}")
    print("Now run the notebook's Cell 0 (it will reuse this setup).")
    print("If FLUX.1-dev is needed, also run: !huggingface-cli login")


if __name__ == "__main__":
    main()


Mounted at /content/drive
$ git clone https://github.com/superpevel/liya_diploma.git /content/drive/MyDrive/liya_diplomCC
$ pip install -q -r /content/drive/MyDrive/liya_diplomCC/requirements.txt
$ git clone https://github.com/ostris/ai-toolkit /content/ai-toolkit
$ pip install -q -r /content/ai-toolkit/requirements.txt


In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/liya_diplomCC'
    AI_TOOLKIT = '/content/ai-toolkit'
    sys.path.insert(0, DRIVE_ROOT)
    sys.path.insert(0, AI_TOOLKIT)
    # Colab-only: install runtime deps. Local installs are handled by setup_local.{ps1,sh}.
    get_ipython().system('pip install -q resvg-py datasets huggingface-hub Pillow tqdm')
except ModuleNotFoundError:
    # Local run: find project root by looking for scripts/ directory
    _here = Path().resolve()
    DRIVE_ROOT = str(_here if (_here / 'scripts').exists() else _here.parent)
    AI_TOOLKIT = str(Path(DRIVE_ROOT).parent / 'ai-toolkit')
    sys.path.insert(0, DRIVE_ROOT)
    sys.path.insert(0, AI_TOOLKIT)

print(f"DRIVE_ROOT: {DRIVE_ROOT}")

SVG_DIR = f'{DRIVE_ROOT}/data/raw_svg'
PNG_DIR = f'{DRIVE_ROOT}/data/png_512'


In [ ]:
# Search huggingface.co/datasets for "svg logo" to find the best available dataset.
# Recommended IDs to try (verify availability before running):
#   "logo-wizard/modern-logo-dataset"
#   "svgcoder/svg-logos" (if available)
# If no SVG dataset is found, use PNG logos and skip the SVG filter step.

from datasets import load_dataset

DATASET_ID = "logo-wizard/modern-logo-dataset"  # UPDATE after checking HuggingFace
ds = load_dataset(DATASET_ID, split="train")
print(f"Loaded {len(ds)} samples")
print("Features:", ds.features)


In [ ]:
from pathlib import Path
from tqdm import tqdm
from PIL import Image

Path(SVG_DIR).mkdir(parents=True, exist_ok=True)
Path(PNG_DIR).mkdir(parents=True, exist_ok=True)

n_svg, n_png = 0, 0
for i, item in enumerate(tqdm(ds)):
    if item.get("svg"):
        # SVG-bearing dataset: drop the raw SVG, the next cell rasterises it.
        with open(f"{SVG_DIR}/{i:06d}.svg", "w", encoding="utf-8") as f:
            f.write(item["svg"])
        n_svg += 1
    elif "image" in item and item["image"] is not None:
        # PNG-only dataset: resize to 512x512 with white-padded square crop and save directly to PNG_DIR.
        img = item["image"].convert("RGB")
        side = max(img.size)
        canvas = Image.new("RGB", (side, side), (255, 255, 255))
        canvas.paste(img, ((side - img.size[0]) // 2, (side - img.size[1]) // 2))
        canvas.resize((512, 512), Image.LANCZOS).save(f"{PNG_DIR}/{i:06d}.png", "PNG")
        n_png += 1

print(f"Saved {n_svg} SVG -> {SVG_DIR}")
print(f"Saved {n_png} PNG -> {PNG_DIR}")


In [ ]:
from pathlib import Path
from scripts.svg_to_png import batch_convert

svg_files = list(Path(SVG_DIR).glob("**/*.svg"))
if svg_files:
    stats = batch_convert(SVG_DIR, PNG_DIR, size=512)
    print(f"Converted: {stats['success']} OK, {stats['failed']} failed of {stats['total']}")
else:
    n_pngs = len(list(Path(PNG_DIR).glob("*.png")))
    print(f"No SVG files in {SVG_DIR} - dataset is PNG-only.")
    print(f"PNG_DIR already has {n_pngs} images, skipping rasterisation.")


In [ ]:
import importlib
import json
import scripts.filter_dataset
importlib.reload(scripts.filter_dataset)
from scripts.filter_dataset import filter_dataset

filtered = filter_dataset(PNG_DIR, SVG_DIR, min_paths=3, max_paths=500)
print(f"After filtering: {len(filtered)} valid pairs")

with open(f'{DRIVE_ROOT}/data/filtered_pairs.jsonl', 'w') as f:
    for item in filtered:
        f.write(json.dumps(item) + '\n')
print("Saved data/filtered_pairs.jsonl")


In [ ]:
import random
import os
from PIL import Image
import matplotlib.pyplot as plt

if not filtered:
    print("No filtered pairs to preview - check earlier cells (PNG_DIR empty or aspect-ratio filter too strict).")
else:
    sample = random.sample(filtered, min(5, len(filtered)))
    fig, axes = plt.subplots(1, len(sample), figsize=(15, 3), squeeze=False)
    axes = axes[0]
    for ax, item in zip(axes, sample):
        ax.imshow(Image.open(item['png_path']))
        ax.axis('off')
    plt.suptitle(f"{len(sample)} random logos from {len(filtered)} filtered pairs")
    plt.tight_layout()
    out_path = f'{DRIVE_ROOT}/results/experiments/dataset_sample.png'
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(out_path, dpi=150)
    plt.show()
